# Low pass filtering

As specified on the datasheet and register map, the MPU6050 has a built-in digital low pass filter. It's helpful because it provides some smoothing, noise reduction and anti-aliasing at zero computational cost on the microcontroller. On the other hand, it introduces a delay/phase shift in the signal, getting worse with lower cutoff frequencies. Unfortunately, the exact characteristics of the filter are not specified. Instead, only the cutoff frequency can be configured by setting the DLPF_CFG bits in the CONFIG register (26). Available frequencies are: 

| DLPF_CFG | Accel Bandwidth (Hz) | Delay (ms) | Gyro Bandwidth (Hz) | Delay (ms) | Fs (kHz) |
|----------|---------------------|------------|----------------------|------------|-----|
| 0 | 256 | 0.98 | 260 | 0.98 | 8 |
| 1 | 188 | 1.9  | 184 | 2.0  | 1 |
| 2 | 98  | 2.8  | 94  | 3.0  | 1 |
| **3** | **42**  | **4.8**  | **44**  | **4.9**  | **1** |
| 4 | 20  | 8.3  | 21  | 8.5  | 1 |
| 5 | 10  | 13.4 | 10  | 13.8 | 1 |
| 6 | 5   | 18.6 | 5   | 19.0 | 1 |
| 7 | RESERVED | RESERVED | RESERVED | RESERVED | 8 |

Other filter characteristics like the order, the type, or the presence of passband ripple or ringing are not specified. It's assumed that it's a digital low order IIR (infinite-impulse response) filter. I think this because the sensor was/is designed to be low power and low cost, and otherwise Invensense would have leveraged the filter characteristics as a selling point.

Based on a brief research and common sense, most common human movements fall comfortably below 10-20Hz, but since device will be used for high speed/power movements and there may be impacts, vibration or resonance, a margin of safety is needed. Additionally, according to the sampling theorem, the sampling frequency should be at least twice the highest frequency of interest to avoid aliasing. Based on this, 42/44Hz seems like a good first choice, trying to match it with a sampling frequency of at least 100Hz ideally. 

To avoid overengineering, initially I thought of just using this built-in filter, but additional filtering is needed since I have seen downstream that velocity is very sensitive to noise. A small, fast software low pass filter like the moving average filter (which is a FIR, finite impulse response) will be tested with different kernel lengths to find a good compromise between noise reduction, responsiveness, and signal sharpness. Here, convolution with a rectangular kernel will be used, but on the device it can be implemented as a fixed-size sliding window, which is more efficient ($O(n)$ instead of $O(k \cdot n)$, where $k$ is the length of the kernel and $n$ is the signal length). This filter has linear phase (i.e. symmetric impulse response that produces a constant delay) and it's a good choice for white noise reduction.

Notice that the alternative on the time domain, a single pole IIR filter, while it requires a similar programming effort than the moving average (minimal), it has a non-linear phase response, which can distort the signal by introducing non-constant delays, and to make it zero-phase two passes of filtering are needed (one forwards and one backwards), which is not possible for the decided streaming, real-time architecture. Because of this reason, it will not be used.

Resources:
* https://valdperformance.com/news/sampling-frequency-how-much-is-enough
* [The scientist and engineer's guide to digital signal processing, Steven W. Smith - Chapters 3, 15, 19, 21](../../docs/misc/The%20Scientist%20and%20Engineers%20Guide%20to%20Digital%20signal%20processing%20-%20Steven%20W.%20Smith.pdf)
* https://www.youtube.com/watch?v=t3fCbQCSeQs
* [MPU6050 datasheet](../../docs/misc/MPU6050%20datasheet.pdf)
* [MPU6050 register map](../../docs/misc/MPU6050%20registers.pdf)

